# Demonstrate the Voxel-based approach

We begin with the imports

In [ ]:
from os import chdir, path
thepath = path.abspath("../base")
chdir(thepath)

In [ ]:
# yaml processing file loading. 
import yaml
import numpy as np
import matplotlib.pyplot as plt

from os.path import exists
from stl import mesh
from MST_centreline import return_MST_centrelines
from utils import return_source_targets, find_neighbours_with_knn, adjust_plane

# to show the various clearance maps. 
from matplotlib.colors import Normalize 
from matplotlib.cm import ScalarMappable
from matplotlib.gridspec import GridSpec

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


A mapping to the conduits and their number for easier access.

In [3]:
with open(f"../notebooks/conduit_map.yaml", "r") as f: 
    conduit_map = yaml.safe_load(f)
    f.close()

A mapping of plotting keyword arguments for the various objects to be plotted. 

In [4]:
with open(f"../notebooks/plot_kwargs.yaml", "r") as f: 
    pl = yaml.safe_load(f)
    f.close()

In [5]:
skeleton_kwargs = pl["skeleton_kwargs"]
node_kwargs = pl["node_kwargs"]
ct_kwargs = pl["ct_kwargs"]
ct1_kwargs = pl["ct1_kwargs"]
ct2_kwargs = pl["ct2_kwargs"]
mesh_kwargs = pl["mesh_kwargs"]
start_kwargs = pl["start_kwargs"]
end_kwargs = pl["end_kwargs"]

In [ ]:
# set up the conduit name and ID. 
ID = 12
CONDUIT_NAME = conduit_map[ID]

thedir = f"../data/{ID}_{CONDUIT_NAME}"

In [ ]:
## import the mesh data. 
mesh_fp =  f"{thedir}/{CONDUIT_NAME}.stl"
data = mesh.Mesh.from_file(mesh_fp)

## where to store the voxel file when generated. 
voxset_fp = f"{thedir}//mst//{CONDUIT_NAME}_voxset.vxl"

In [ ]:
## import the triangle adjacency matrix if it exists, otherwise compute it (may take a while)
adj_triangles_fp = mesh_fp[:-4] + "_adjacent_triangles.txt"
if not exists(adj_triangles_fp):
    triangle_adjacency_matrix = find_neighbours_with_knn(data)
    np.savetxt(adj_triangles_fp, triangle_adjacency_matrix)
else: 
    triangle_adjacency_matrix = np.loadtxt(adj_triangles_fp)

In [ ]:
with open(f"{thedir}/{CONDUIT_NAME}_processing.yaml", "r") as f: 
    process_p = yaml.safe_load(f)

    f.close()

We read the processing parameters file to find the chosen voxel size.

In [ ]:
voxelsize = process_p["mst"]["voxelSize"]

In [ ]:
P1,_,P2,_ = return_source_targets(data, triangle_adjacency_matrix, eps = voxelsize)

In [ ]:

out_MST = return_MST_centrelines(mesh_fp, voxset_path=voxset_fp, 
                             delta = int(voxelsize*1000), 
                             P1=P1, 
                             P2=P2, 
                             demo= True,
                             verbose = False)


For demonstration purposes, we load the edges of voxels, the edges of the minimum spanning tree, the buffer edges. To colour the edges of the voxels by the clearance that was calculated so far, we also load the coordinates of the mid-points and colour them by clearance.

In [ ]:
mst_centreline = out_MST["centrespline-direct"]

In [ ]:
edges = np.loadtxt(f"{thedir}//mst//{CONDUIT_NAME}_edges.xyz")
edges_mask = np.logical_and(edges[:, 2] == 0,edges[:, 5] == 0) 

In [ ]:
vmin = np.min(edges[:, 6]/1000)
vmax = np.max(edges[:, 6]/1000)

norm = Normalize(vmin, vmax)
mappable = ScalarMappable(norm)

edges_midpoints = np.loadtxt(f"{thedir}//mst//{CONDUIT_NAME}_edges_midpoints.xyz")

In [ ]:
# buffer edges grown around the minimum spanning tree. 
buffer_edges = np.loadtxt(f"{thedir}//mst//{CONDUIT_NAME}_unzigzag_edges.xyz")
buffer_edges_mask = np.logical_and(buffer_edges[:, 2] == 0, buffer_edges[:, 5] == 0) 

buffer_centreline = np.loadtxt(f"{thedir}//mst//{CONDUIT_NAME}_{int(voxelsize*1000)}_jagged_centreline.txt")

mst_edges = np.loadtxt(f"{thedir}//mst//{CONDUIT_NAME}_MST_edges.xyz")
mst_edges_mask = np.logical_and(edges[:, 2] == 0,edges[:, 5] == 0) 

In [ ]:
smooth_centreline = np.loadtxt(f"{thedir}//mst//{CONDUIT_NAME}_{int(voxelsize*1000)}_centreline.txt")

In [ ]:
gs = GridSpec(9, 40)
fig = plt.figure(figsize = (13, 6))

##  the voxel clearance map figure. 
ax = fig.add_subplot(gs[0:4, :13])
for edge, edge_mask in zip(edges, edges_mask):
    if edge_mask:
        ax.plot(edge[[0,3]],edge[[1,4]], color = "grey", zorder = -25, lw = 0.5)


# plot the mesh only when centroids z coordinates are close to 
mesh_mask = np.abs(data.centroids[:, 2]) < 0.05
ax.scatter(*data.centroids[mesh_mask][:, [0, 1]].T, 
        **mesh_kwargs)

# plot voxel edges when their z coordinate is close to 0.
mask = np.abs(edges_midpoints[:, 2]) < 0.05
ax.scatter(*edges_midpoints[mask][:, [0, 1]].T, c=edges_midpoints[mask, 3], zorder = -10, marker="s", s= 2)

# set up the colorbar.
cax = fig.add_subplot(gs[0:4, 14:15])
cb = plt.colorbar(mappable, cax =cax, fraction=0.01)
cb.set_label("clearance")
ax.set_aspect("equal")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("a)", fontdict= dict(fontsize=9))

## unzizag path plot. 
ax = fig.add_subplot(gs[5:, :13])
for edge, edge_mask in zip(edges, edges_mask):
    if edge_mask:
        ax.plot(edge[[0,3]],edge[[1,4]], color = "grey", zorder = -25, lw = 0.5)
ax.plot(edges[0,[0,3]],edges[0,[1,4]], color = "grey", zorder = -25, lw = 0.5, label ="voxel edge")

# plot the mesh 
mesh_mask = np.abs(data.centroids[:, 2]) < 0.05
ax.scatter(*data.centroids[mesh_mask][:, [0, 1]].T, 
        **mesh_kwargs)


ax.set_title("b)", fontdict= dict(fontsize=9))

# plot the jagged centreline
ax.plot(*buffer_centreline[:, [0, 1]].T, zorder = 100, color = "k", label = "MST path")
ax.plot(*buffer_centreline[:, [0, 1]].T, zorder = 95, color = "white", lw = 1.5)

for buffer_edge, buffer_edge_mask in zip(buffer_edges, buffer_edges_mask):
    if buffer_edge_mask:
        ax.plot(buffer_edge[[0,3]],buffer_edge[[1,4]], color = "k", zorder = -10, lw = 0.5)

ax.plot(buffer_edges[0, [0,3]],buffer_edges[0, [1,4]], color = "k", zorder = -10, lw = 0.5, label ="buffer edges")


# plot the source and target
ax.scatter(*P1[[0, 1]],**start_kwargs)
ax.scatter(*P2[[0, 1]], **end_kwargs)

# define the colorbar for the mesh midpoints. 
ax.legend(loc = "upper left", bbox_to_anchor= (1,1.02), fontsize = 9)
ax.set_aspect("equal")

ax.set_xlabel("x")
ax.set_ylabel("y")

ax = fig.add_subplot(gs[0:4, 21:-6])
for edge, edge_mask in zip(edges, edges_mask):
    if edge_mask:
        ax.plot(edge[[0,3]],edge[[1,4]], color = "grey", zorder = -20, lw = 0.5)
ax.plot(edges[0,[0,3]],edges[0,[1,4]], color = "grey", zorder = -25, lw = 0.5, label ="voxel edge")


# plot the centreline
ax.plot(*np.array(out_MST["raw"])[:, [0, 1]].T,**ct_kwargs, label = "raw centreline")
# plot the centreline nodes
ax.scatter(*np.array(out_MST["raw"])[:, [0, 1]].T, 
        **node_kwargs)
for buffer_edge, buffer_edge_mask in zip(buffer_edges, buffer_edges_mask):
    if buffer_edge_mask:
        ax.plot(buffer_edge[[0,3]],buffer_edge[[1,4]], color = "k", zorder = -10, lw = 0.25)


ax.set_title("c)", fontdict= dict(fontsize=9))

# plot the source and target
ax.scatter(*P1[[0, 1]],**start_kwargs)
ax.scatter(*P2[[0, 1]], **end_kwargs)


# plot the mesh 
mesh_mask = np.abs(data.centroids[:, 2]) < 0.05
ax.scatter(*data.centroids[mesh_mask][:, [0, 1]].T, 
        **mesh_kwargs)

ax.legend(loc = "upper left", bbox_to_anchor= (1,1.02), fontsize = 9)
ax.set_aspect("equal")

ax.set_xlabel("x")
ax.set_ylabel("y")


ax = fig.add_subplot(gs[5:, 21:-6])


# plot the centreline
ax.plot(*np.array(out_MST["raw"])[:, [0, 1]].T,**ct_kwargs, label = "raw centreline")
# plot the centreline nodes
ax.plot(*np.array(out_MST["simplified"])[:, [0, 1]].T,**ct1_kwargs, label = "douglas-peucker-simplified")
ax.plot(*np.array(out_MST["centerspline_direct"])[:, [0, 1]].T,**ct2_kwargs, label = "spline based on nodes")

ax.set_title("d)", fontdict= dict(fontsize=9))

# plot the source and target
ax.scatter(*P1[[0, 1]],**start_kwargs)
ax.scatter(*P2[[0, 1]], **end_kwargs)


# plot the mesh 
mesh_mask = np.abs(data.centroids[:, 2]) < 0.05
ax.scatter(*data.centroids[mesh_mask][:, [0, 1]].T, 
        **mesh_kwargs)

ax.legend(loc = "upper left", bbox_to_anchor= (1,1.02), fontsize = 9)
ax.set_aspect("equal")

ax.set_xlabel("x")
ax.set_ylabel("y")


plt.savefig(f"{thedir}//mst//{CONDUIT_NAME}_edge_clearance_map.png", dpi = 300)
plt.savefig(f"{thedir}//mst/{CONDUIT_NAME}_edge_clearance_map.pdf")

plt.show()